In [1]:
# Tottenham,Newcastle
# Aston Villa,Leicester
# Bournemouth,Everton
# Crystal Palace,Chelsea
# Manchester City,West Ham
# Southampton,Brentford
# Brighton,Arsenal
# Fulham,Ipswich
# Liverpool,Manchester United
# Wolves,Nottingham Forest

In [2]:
import requests
import pandas as pd
import json
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

In [3]:
gw = 32

fixtures = requests.get(f'https://fantasy.premierleague.com/api/fixtures/?event={gw}').json()
boots_ = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/').json()

fixture_dets =  [{'team_a': fix['team_a'], 'team_h': fix['team_h'], 'h_fdr': fix['team_h_difficulty'], 'a_fdr': fix['team_a_difficulty'] } for fix in fixtures]
fix_df = pd.DataFrame(fixture_dets)

team_ids = [{'team_code': team['code'], 'team_name': team['name'], 'team_id': team['id']} for team in boots_['teams']]
id_df = pd.DataFrame(team_ids)

def add_team_name(row):
    away = row['team_a']
    home = row['team_h']

    home_team_row = id_df[id_df['team_id'] == home]['team_name'].values
    away_team_row = id_df[id_df['team_id'] == away]['team_name'].values

    team_home = home_team_row[0] if home_team_row.size > 0 else 'None'
    team_away = away_team_row[0] if away_team_row.size > 0 else 'None'

    if(team_home == 'Man Utd'):
        team_home = 'Manchester United'
    elif(team_away == 'Man Utd'):
        team_away = 'Manchester United'

    if(team_home == 'Man City'):
        team_home = 'Manchester City'
    elif(team_away == 'Man City'):
        team_away = 'Manchester City'

    if(team_home == 'Wolves'):
        team_home = 'Wolverhampton Wanderers'
    elif(team_away == 'Wolves'):
        team_away = 'Wolverhampton Wanderers'

    if(team_home == "Nott'm Forest"):
        team_home = "Nottingham Forest"
    elif(team_away == "Nott'm Forest"):
        team_away = "Nottingham Forest"

    if(team_home == "Spurs"):
        team_home = "Tottenham"
    elif(team_away == "Spurs"):
        team_away = "Tottenham"

    if(team_home == "Newcastle"):
        team_home = "Newcastle United"
    elif(team_away == "Newcastle"):
        team_away = "Newcastle United"


    return pd.Series([team_home, team_away])


fix_df[['team_home', 'team_away']] = fix_df.apply(add_team_name, axis=1)

fix_df.to_csv(f'./fpl_team_ids_{gw}.csv', index=False)


In [4]:
pd.read_csv(f'./fpl_team_ids_{gw}.csv')

,team_a,team_h,h_fdr,a_fdr,team_home,team_away
0,7,13,3,4,Manchester City,Crystal Palace
1,11,5,2,3,Brighton,Leicester
2,8,16,2,4,Nottingham Forest,Everton
3,2,17,3,2,Southampton,Aston Villa
4,4,1,2,4,Arsenal,Brentford
5,10,6,2,4,Chelsea,Ipswich
6,19,12,2,5,Liverpool,West Ham
7,18,20,3,2,Wolverhampton Wanderers,Tottenham
8,14,15,3,4,Newcastle United,Manchester United
9,9,3,3,3,Bournemouth,Fulham


In [5]:
premLeague = "https://sports.williamhill.com/betting/en-gb/football/competitions/OB_TY295/English-Premier-League/matches/OB_MGMB/Match-Betting"

driver = webdriver.Firefox()
driver.get(premLeague)
matches = driver.find_elements(By.CSS_SELECTOR, "article.sp-o-market--default")[0:10]
details = pd.DataFrame({'h_team':[], 'a_team':[], 'WHH':[], 'WHD':[], 'WHA':[]})

# Get links for the matches
# [[match, link]]
for match in matches:
    # matches_links.append([match.text, match.get_attribute('href')])
    # Extract teams
    teams = match.find_element(By.CSS_SELECTOR, 'main.sp-o-market__title span').text
    odds_els = match.find_elements(By.CSS_SELECTOR, 'section.sp-o-market__buttons .sp-betbutton > span')

    # Extract odds
    # odds = [round(int(btn.text.split('/')[0]) / int(btn.text.split('/')[1]) +1, 2)   for btn in match.find_elements(By.CSS_SELECTOR, 'section.sp-o-market__buttons .sp-betbutton > span')]

    odds = []
    for btn in odds_els:
        if btn.text == 'EVS':
            odds.append(round(2.0, 2))
        else:
            odds.append(round(int(btn.text.split('/')[0]) / int(btn.text.split('/')[1]) +1, 2))
    h_odds = 1/odds[0]
    d_odds = 1/odds[1]
    a_odds = 1/odds[2]

    sum_odd_probs = h_odds + d_odds + a_odds
    match_details = {
        'h_team': [teams.split(' v ')[0]],
        'a_team': [teams.split(' v ')[1]],
        'WHH': [round(h_odds/sum_odd_probs, 3)],
        'WHD': [round(d_odds/sum_odd_probs, 3)],
        'WHA': [round(a_odds/sum_odd_probs, 3)]
        }
    match_dets_df = pd.DataFrame(match_details)
    details = pd.concat([details, match_dets_df], ignore_index=True)
    # print(odds)
    # print({'Home': teams.split(' v ')[0], 'Away': teams.split(' v ')[1], 'WHH': odds[0], 'WHD': odds[1], 'WHA': odds[2]  })


driver.close()


details

,h_team,a_team,WHH,WHD,WHA
0,Man City,Crystal Palace,0.624,0.212,0.164
1,Brighton,Leicester,0.732,0.173,0.095
2,Nottingham Forest,Everton,0.449,0.290,0.262
3,Southampton,Aston Villa,0.187,0.216,0.596
4,Arsenal,Brentford,0.615,0.220,0.164
5,Chelsea,Ipswich,0.695,0.188,0.117
6,Liverpool,West Ham,0.703,0.187,0.110
7,Wolves,Tottenham,0.413,0.271,0.316
8,Newcastle,Man Utd,0.542,0.243,0.215
9,Bournemouth,Fulham,0.441,0.263,0.296


In [6]:

# Manchester City,
# Manchester United
def update_name(row):
    if(row['h_team'] == 'Man Utd'):
        row['h_team'] = 'Manchester United'
    elif(row['a_team'] == 'Man Utd'):
        row['a_team'] = 'Manchester United'

    if(row['h_team'] == 'Man City'):
        row['h_team'] = 'Manchester City'
    elif(row['a_team'] == 'Man City'):
        row['a_team'] = 'Manchester City'

    if(row['h_team'] == 'Wolves'):
        row['h_team'] = 'Wolverhampton Wanderers'
    elif(row['a_team'] == 'Wolves'):
        row['a_team'] = 'Wolverhampton Wanderers'

    if(row['h_team'] == 'Newcastle'):
        row['h_team'] = 'Newcastle United'
    elif(row['a_team'] == 'Newcastle'):
        row['a_team'] = 'Newcastle United'
    return row


details_ = details.apply(update_name, axis=1)
details_

,h_team,a_team,WHH,WHD,WHA
0,Manchester City,Crystal Palace,0.624,0.212,0.164
1,Brighton,Leicester,0.732,0.173,0.095
2,Nottingham Forest,Everton,0.449,0.290,0.262
3,Southampton,Aston Villa,0.187,0.216,0.596
4,Arsenal,Brentford,0.615,0.220,0.164
5,Chelsea,Ipswich,0.695,0.188,0.117
6,Liverpool,West Ham,0.703,0.187,0.110
7,Wolverhampton Wanderers,Tottenham,0.413,0.271,0.316
8,Newcastle United,Manchester United,0.542,0.243,0.215
9,Bournemouth,Fulham,0.441,0.263,0.296


In [7]:

details_.to_csv(f'./odds_{gw}.csv', index=False)

## Add fdr


In [8]:

odds_ = pd.read_csv(f'./odds_{gw}.csv')
teams = pd.read_csv(f'./fpl_team_ids_{gw}.csv')

def add_fdr(row):
    team_row = teams[teams['team_home'] == row['h_team']]
    h_fdr = team_row['h_fdr'].values[0] if team_row['h_fdr'].values.size > 0 else 'None'
    a_fdr = team_row['a_fdr'].values[0] if team_row['a_fdr'].values.size > 0 else 'None'

    return pd.Series([h_fdr, a_fdr])

odds_[['h_fdr', 'a_fdr']] = odds_.apply(add_fdr, axis=1)
odds_.to_csv(f'./odds_{gw}.csv', index=False)
odds_

,h_team,a_team,WHH,WHD,WHA,h_fdr,a_fdr
0,Manchester City,Crystal Palace,0.624,0.212,0.164,3,4
1,Brighton,Leicester,0.732,0.173,0.095,2,3
2,Nottingham Forest,Everton,0.449,0.290,0.262,2,4
3,Southampton,Aston Villa,0.187,0.216,0.596,3,2
4,Arsenal,Brentford,0.615,0.220,0.164,2,4
5,Chelsea,Ipswich,0.695,0.188,0.117,2,4
6,Liverpool,West Ham,0.703,0.187,0.110,2,5
7,Wolverhampton Wanderers,Tottenham,0.413,0.271,0.316,3,2
8,Newcastle United,Manchester United,0.542,0.243,0.215,3,4
9,Bournemouth,Fulham,0.441,0.263,0.296,3,3


## Update the team names of the odds


In [9]:
import os

In [10]:
prev_odds = pd.read_csv(os.path.abspath("./E0 24-25.csv")) # Does this work

prev_odds = prev_odds.rename(columns={'HomeTeam': 'h_team', 'AwayTeam': 'a_team'})

def update_name(row):
    # print(row['h_team'], row['a_team'])
    if(row['h_team'] == 'Man United'):
        row['h_team'] = 'Manchester United'
    elif(row['a_team'] == 'Man United'):
        row['a_team'] = 'Manchester United'

    if(row['h_team'] == 'Man City'):
        row['h_team'] = 'Manchester City'
    elif(row['a_team'] == 'Man City'):
        row['a_team'] = 'Manchester City'

    if(row['h_team'] == 'Wolves'):
        row['h_team'] = 'Wolverhampton Wanderers'
    elif(row['a_team'] == 'Wolves'):
        row['a_team'] = 'Wolverhampton Wanderers'

    if(row['h_team'] == 'Newcastle'):
        row['h_team'] = 'Newcastle United'
    elif(row['a_team'] == 'Newcastle'):
        row['a_team'] = 'Newcastle United'

    if(row['h_team'] == "Nott'm Forest"):
        row['h_team'] = 'Nottingham Forest'
    elif(row['a_team'] == "Nott'm Forest"):
        row['a_team'] = 'Nottingham Forest'
    return row


prev_odds = prev_odds.apply(update_name, axis=1)

prev_odds.to_csv("./E0 24-25.csv", index=False)
